<a href="https://colab.research.google.com/github/BoyaWangCambridge/Barra-Factor-Model-with-Orthogonalised-ESG/blob/main/company_controversy_monitor_google_rss_reviewed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Company Controversy News Monitor
This notebook:
1. discovers corporate controversy stories through **Google News RSS** (no SerpAPI key or search credits required);
2. retrieves the main article text when the publisher page is accessible;
3. falls back to the RSS headline and summary when extraction fails;
4. detects controversy themes and estimates severity using a transparent dictionary plus an optional zero-shot model;
5. identifies company candidates with Hugging Face GLiNER (with spaCy fallback) and ranks likely involvement;
6. estimates article tone with a multilingual Hugging Face sentiment model, with VADER fallback;
7. optionally discovers article topics using LDA;
8. exports article, company, and quality-control outputs to Excel.

Limitations:
- Google News RSS usually provides headlines and summaries, not guaranteed full article text.
- Google may throttle RSS requests (HTTP 429), especially from shared Colab IP addresses. This notebook uses delays and limited retries, but cannot guarantee uninterrupted access.
- Full-text extraction can fail because of paywalls, JavaScript, anti-bot controls, SSL restrictions, or publisher access rules. The notebook does not bypass access controls.
- **Without a predefined issuer universe, extracted names are company candidates**. GLiNER NER confidence measures entity extraction confidence, not whether a company committed misconduct. Review evidence and involvement scores before using results for investment decisions.
- The main reason this corporate controversy monitor is slow is that it combines many sequential web requests with several computationally expensive NLP models. It is not just downloading news: it is trying to resolve links, extract article text, identify companies, classify controversy themes, estimate severity, and analyse sentiment.


### Preparation

In [36]:
#  Install Python packages (libraries) from PyPI
%pip install pandas numpy requests beautifulsoup4 lxml feedparser nltk spacy \
     scikit-learn openpyxl trafilatura gliner transformers torch sentencepiece \
    googlenewsdecoder "selectolax<1.0" nltk


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 20.7 MB/s  0:00:13:00:0100:01

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [38]:
# Standard Python
import os
import re
import time
import random
import shutil

from pathlib import Path
from collections import defaultdict
from html import unescape
from urllib.parse import quote_plus
from getpass import getpass

# Data analysis
import numpy as np
import pandas as pd

# Web scraping & news
import requests
import feedparser
from bs4 import BeautifulSoup

# NLP
import nltk
import spacy
from nltk.sentiment import SentimentIntensityAnalyzer

# Topic modelling
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Article extraction
try:
    import trafilatura
    USE_TRAFILATURA = True
except ImportError:
    USE_TRAFILATURA = False
    print("Trafilatura not available. Using BeautifulSoup fallback.")


### Model setup: spaCy + Hugging Face GLiNER

This notebook uses spaCy for sentence segmentation and dependency parsing, and Hugging Face GLiNER to identify company entities with custom labels. GLiNER is configured to distinguish companies from regulators, government agencies, NGOs, and news organisations. **Google News RSS is used for discovery**, Trafilatura is preferred for article extraction, and optional multilingual sentiment and zero-shot models provide additional tone, theme, and severity signals.

In Google Colab, install the dependencies in a separate cell before running the notebook. The spaCy English model is installed separately with `!python -m spacy download en_core_web_lg`. Models may download on first use and may take several minutes. If a model cannot load, the notebook attempts the available fallback path.


In [39]:
from pathlib import Path

import numpy as np
import spacy

from nltk.sentiment import SentimentIntensityAnalyzer


# Load spaCy for sentence segmentation and dependency parsing.
# Keep the large English model because the scoring function uses dependencies.
nlp = spacy.load("en_core_web_lg")

# ------------------------------------------------------------
# Hugging Face GLiNER company entity recognition
# ------------------------------------------------------------
# Install once if needed in a separate notebook cell:
# %pip install gliner
#
# This multilingual model supports custom entity labels. It is
# used to identify company-like entities separately from regulators,
# government agencies, NGOs and news organisations.

GLINER_MODEL_NAME = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50

GLINER_ENTITY_LABELS = [
    "company",
    "corporation",
    "business organization",
    "government agency",
    "regulatory body",
    "non-profit organization",
    "news organization",
]

COMPANY_ENTITY_LABELS = {
    "company",
    "corporation",
    "business organization",
}

try:
    from gliner import GLiNER

    company_ner = GLiNER.from_pretrained(GLINER_MODEL_NAME)
    USE_GLINER = True
    print(f"GLiNER loaded: {GLINER_MODEL_NAME}")

except Exception as error:
    company_ner = None
    USE_GLINER = False
    print(
        "GLiNER could not be loaded; spaCy ORG fallback will be used. "
        "Check whether `gliner` is installed and Hugging Face is accessible."
    )
    print(f"{type(error).__name__}: {error}")

# ------------------------------------------------------------
# Hugging Face tone and controversy classifiers
# ------------------------------------------------------------
# Install if needed in a separate cell:
# %pip install transformers torch sentencepiece
#
# Multilingual sentiment:
#   predicts positive / neutral / negative tone.
#
# Multilingual zero-shot classifier:
#   helps identify themes and estimate impact severity.
#
# These are screening models, not validated ESG ratings.
# Model download may fail behind corporate firewalls.

ENABLE_HF_SENTIMENT = True
ENABLE_HF_ZERO_SHOT = False ### Change

HF_SENTIMENT_MODEL = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
HF_ZERO_SHOT_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

tone_pipeline = None
zero_shot_pipeline = None
USE_HF_SENTIMENT = False
USE_HF_ZERO_SHOT = False

if ENABLE_HF_SENTIMENT:
    try:
        from transformers import pipeline
        tone_pipeline = pipeline(
            "sentiment-analysis",
            model=HF_SENTIMENT_MODEL
        )
        USE_HF_SENTIMENT = True
        print(f"Multilingual sentiment model loaded: {HF_SENTIMENT_MODEL}")
    except Exception as error:
        print("Multilingual sentiment model unavailable; will use VADER if available.")
        print(f"{type(error).__name__}: {error}")

if ENABLE_HF_ZERO_SHOT:
    try:
        from transformers import pipeline
        zero_shot_pipeline = pipeline(
            "zero-shot-classification",
            model=HF_ZERO_SHOT_MODEL
        )
        USE_HF_ZERO_SHOT = True
        print(f"Zero-shot classifier loaded: {HF_ZERO_SHOT_MODEL}")
    except Exception as error:
        print("Zero-shot classifier unavailable; dictionary-based theme/severity fallback will be used.")
        print(f"{type(error).__name__}: {error}")


# Replace this with your actual file path
VADER_FILE = Path(
    r"C:\Users\bwang\Desktop\Corporate Controversy and Engagement\vader_lexicon.txt"
)

CLEAN_VADER_FILE = VADER_FILE.with_name(
    "vader_lexicon_clean.txt"
)


def clean_vader_lexicon(
    input_file,
    output_file
):
    valid_lines = []
    invalid_lines = []

    with input_file.open(
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):
            line = line.strip()

            # Ignore empty lines
            if not line:
                continue

            columns = line.split("\t")

            # VADER needs at least:
            # word<TAB>sentiment score
            if len(columns) < 2:
                invalid_lines.append(
                    (line_number, line[:100])
                )
                continue

            word = columns[0].strip()
            score = columns[1].strip()

            if not word:
                invalid_lines.append(
                    (line_number, line[:100])
                )
                continue

            # Confirm that the score is numeric
            try:
                float(score)

            except ValueError:
                invalid_lines.append(
                    (line_number, line[:100])
                )
                continue

            # NLTK only needs the first two columns
            valid_lines.append(
                f"{word}\t{score}"
            )

    if not valid_lines:
        raise ValueError(
            "No valid VADER lexicon entries were found. "
            "The selected file may be an HTML webpage "
            "rather than the raw lexicon text file."
        )

    # A final newline is normally safe, but the current NLTK
    # parser can fail on empty rows, so do not add one here.
    output_file.write_text(
        "\n".join(valid_lines),
        encoding="utf-8"
    )

    print(
        f"Valid lexicon entries: "
        f"{len(valid_lines):,}"
    )

    print(
        f"Invalid lines removed: "
        f"{len(invalid_lines):,}"
    )

    if invalid_lines:
        print("\nFirst invalid lines:")

        for line_number, content in invalid_lines[:10]:
            print(
                f"Line {line_number}: "
                f"{content!r}"
            )

    return output_file

/usr/local/python/3.14.2/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 865.98it/s]
[transformers] Could not extract SentencePiece model from /home/codespace/.cache/huggingface/hub/models--microsoft--mdeberta-v3-base/snapshots/a0484667b22365f84929a935b5e50a51f71f159d/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


GLiNER could not be loaded; spaCy ORG fallback will be used. Check whether `gliner` is installed and Hugging Face is accessible.
ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5749.71it/s]


Multilingual sentiment model loaded: lxyuan/distilbert-base-multilingual-cased-sentiments-student


In [40]:
# Set up VADER

from pathlib import Path
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# File locations
VADER_FILE = Path("./vader_lexicon.txt")
CLEAN_VADER_FILE = Path("./vader_lexicon_clean.txt")

# Helper function
def clean_vader_lexicon(source_file, output_file):
    with open(source_file, "r", encoding="utf-8") as src:
        lines = src.readlines()

    with open(output_file, "w", encoding="utf-8") as dst:
        for line in lines:
            line = line.strip()
            if line:
                dst.write(line + "\n")

print("VADER setup complete")

VADER setup complete


In [41]:
# Load VADER from the optional custom lexicon if available; otherwise use NLTK's lexicon.
sia = None
USE_VADER = False

if VADER_FILE.exists():
    try:
        clean_vader_lexicon(VADER_FILE, CLEAN_VADER_FILE)
        sia = SentimentIntensityAnalyzer(
            lexicon_file=CLEAN_VADER_FILE.resolve().as_uri()
        )
        USE_VADER = True
        print("VADER loaded from custom lexicon:", CLEAN_VADER_FILE)

    except Exception as error:
        print(
            "Custom VADER lexicon could not be loaded; "
            "trying the standard NLTK lexicon."
        )
        print(f"{type(error).__name__}: {error}")

if not USE_VADER:
    try:
        nltk.download("vader_lexicon", quiet=True)
        sia = SentimentIntensityAnalyzer()
        USE_VADER = True
        print("VADER loaded using the standard NLTK lexicon.")
    except Exception as error:
        sia = None
        USE_VADER = False
        print("VADER unavailable; tone analysis will rely on Hugging Face if loaded.")
        print(f"{type(error).__name__}: {error}")


Custom VADER lexicon could not be loaded; trying the standard NLTK lexicon.
PermissionError: Security Violation [pathsec.open]: Unauthorized path /workspaces/Barra-Factor-Model-with-Orthogonalised-ESG/vader_lexicon_clean.txt
VADER loaded using the standard NLTK lexicon.


In [42]:
if USE_VADER:

    test_text = (
        "The company faces a serious "
        "fraud investigation and regulatory fine."
    )

    print(
        sia.polarity_scores(test_text)
    )

{'neg': 0.395, 'neu': 0.465, 'pos': 0.14, 'compound': -0.5106}


In [43]:
# Settings
GOOGLE_NEWS_LANGUAGE = "en-GB"
GOOGLE_NEWS_COUNTRY = "GB"
GOOGLE_NEWS_CEID = "GB:en"
MAX_RESULTS_PER_QUERY = 10 ### Change

# RSS has no SerpAPI API key or search-credit requirement, but Google may throttle requests.
MAX_RSS_RETRIES = 2
RSS_RETRY_BASE_SECONDS = 20
MIN_SEARCH_DELAY_SECONDS = 5.0
MAX_SEARCH_DELAY_SECONDS = 10.0

REQUEST_TIMEOUT = 25
ARTICLE_DELAY_SECONDS = 1.0
MIN_ARTICLE_LENGTH = 200
MIN_INVOLVEMENT_SCORE = 6
MIN_COMPANY_CHARACTERS = 3

# Model thresholds. Tune these on manually reviewed examples.
GLINER_THRESHOLD = 0.50
ZERO_SHOT_THEME_THRESHOLD = 0.35
ZERO_SHOT_SEVERITY_THRESHOLD = 0.40

NUMBER_OF_LDA_TOPICS = 8
NUMBER_OF_TOPIC_WORDS = 10

OUTPUT_FILE = "company_controversy_monitor.xlsx"


In [44]:
# Google News RSS search queries (English terms)
# Start with a small subset if Google throttles requests, then expand gradually.
CONTROVERSY_SEARCH_QUERIES = [
    '"corporate fraud"',
    '"accounting fraud" company',
    '"money laundering" company',
    'company bribery corruption',
    'company regulatory investigation',
    'company criminal charges',
    'company lawsuit controversy',
    'company antitrust investigation',
    'company environmental violation',
    'company pollution oil spill',
    'company greenwashing investigation',
    'company human rights violation',
    'company forced labour',
    'company child labour',
    'company worker safety',
    'company workplace fatality',
    'company discrimination lawsuit',
    'company product recall safety',
    'company data breach privacy',
    'company sanctions violation',
]


## Controversy dictionary

Severity scale:

- **5 Critical**: criminal misconduct, deaths, forced labour, major environmental harm
- **4 High**: serious violations, fines, recalls, breaches, corruption
- **3 Moderate**: investigations, lawsuits, formal complaints
- **2 Emerging**: concerns, criticism, allegations

In [45]:
CONTROVERSY_DICTIONARY = {
    "Environment": {
        5: ["environmental disaster", "catastrophic pollution", "major oil spill", "toxic spill", "mass wildlife deaths", "illegal deforestation", "severe water contamination"],
        4: ["oil spill", "toxic emissions", "environmental violation", "hazardous waste", "water contamination", "air pollution", "biodiversity destruction", "illegal dumping", "pollution fine"],
        3: ["environmental investigation", "environmental lawsuit", "emissions breach", "pollution complaint", "regulatory environmental probe", "deforestation allegation"],
        2: ["environmental criticism", "climate concern", "community opposition", "greenwashing allegation", "misleading environmental claims"],
    },
    "Climate": {
        5: ["climate fraud", "systematic emissions falsification", "large-scale emissions manipulation"],
        4: ["emissions cheating", "carbon accounting fraud", "climate disclosure violation", "greenwashing"],
        3: ["climate lawsuit", "emissions investigation", "net zero investigation", "carbon disclosure complaint"],
        2: ["climate criticism", "net zero criticism", "transition plan concern", "emissions target concern"],
    },
    "Human Rights": {
        5: ["forced labour", "forced labor", "child labour", "child labor", "modern slavery", "human trafficking", "human rights abuse", "crimes against humanity"],
        4: ["indigenous rights violation", "serious human rights violation", "supply chain labour abuse", "supply chain labor abuse", "migrant worker abuse", "exploitative working conditions"],
        3: ["human rights investigation", "human rights lawsuit", "labour rights investigation", "labor rights investigation", "supply chain investigation"],
        2: ["human rights allegation", "labour rights complaint", "labor rights complaint", "community rights concern"],
    },
    "Labour and Employment": {
        5: ["multiple worker deaths", "fatal industrial accident", "factory collapse", "mine collapse", "workplace fatality"],
        4: ["unsafe working conditions", "serious workplace injury", "systematic wage theft", "union busting", "workplace discrimination", "sexual harassment lawsuit"],
        3: ["labour lawsuit", "labor lawsuit", "worker safety investigation", "employment tribunal", "unfair dismissal", "wage dispute", "worker strike"],
        2: ["employee complaint", "workplace criticism", "labour dispute", "labor dispute", "staff protest"],
    },
    "Corporate Governance": {
        5: ["accounting fraud", "financial fraud", "money laundering", "embezzlement", "bribery", "criminal conspiracy", "insider trading", "market manipulation", "tax evasion"],
        4: ["corruption", "accounting irregularities", "financial misstatement", "sanctions violation", "governance failure", "executive misconduct", "regulatory fine"],
        3: ["fraud investigation", "regulatory investigation", "governance investigation", "shareholder lawsuit", "regulatory probe", "whistleblower complaint"],
        2: ["governance concern", "executive criticism", "shareholder complaint", "board independence concern"],
    },
    "Competition and Business Ethics": {
        5: ["criminal cartel", "systematic price fixing", "large-scale bid rigging"],
        4: ["price fixing", "bid rigging", "anti-competitive conduct", "anticompetitive conduct", "abuse of market dominance", "cartel activity"],
        3: ["antitrust investigation", "competition investigation", "competition lawsuit", "regulatory competition probe"],
        2: ["competition concern", "market dominance criticism", "unfair business practice"],
    },
    "Product Safety and Customers": {
        5: ["product caused deaths", "fatal product defect", "mass consumer harm", "deadly product failure"],
        4: ["major product recall", "serious product defect", "consumer safety violation", "unsafe product", "medical product failure"],
        3: ["product recall", "product safety investigation", "consumer lawsuit", "mis-selling", "false advertising", "customer compensation"],
        2: ["customer complaint", "product criticism", "quality concern", "consumer concern"],
    },
    "Data Privacy and Cybersecurity": {
        5: ["massive data breach", "large-scale identity theft", "criminal misuse of personal data"],
        4: ["data breach", "privacy violation", "customer data leak", "unauthorised data sharing", "unauthorized data sharing", "cybersecurity failure"],
        3: ["privacy investigation", "data protection investigation", "cybersecurity investigation", "privacy lawsuit", "data protection fine"],
        2: ["privacy concern", "data security concern", "customer data complaint"],
    },
    "Legal and Regulatory": {
        5: ["criminal charges", "criminal conviction", "guilty plea", "indicted", "convicted", "prosecuted for fraud"],
        4: ["regulatory sanction", "major fine", "licence revoked", "license revoked", "regulatory violation"],
        3: ["lawsuit", "legal action", "investigation", "regulatory inquiry", "court proceedings", "regulatory probe"],
        2: ["allegation", "legal complaint", "regulatory concern"],
    },
}

In [46]:
# Filters and involvement terms
NON_COMPANY_ORGANISATIONS = {
    "sec", "securities and exchange commission", "us securities and exchange commission",
    "u.s. securities and exchange commission", "fca", "financial conduct authority",
    "ftc", "federal trade commission", "doj", "department of justice", "cma",
    "competition and markets authority", "epa", "environmental protection agency",
    "fbi", "serious fraud office", "sfo", "un", "united nations", "european union",
    "european commission", "european parliament", "world bank",
    "international monetary fund", "imf", "world health organization",
    "world health organisation", "who", "white house", "uk government",
    "u.s. government", "us government", "supreme court", "high court",
    "court of appeal", "parliament", "congress", "senate", "police",
}

INVALID_COMPANY_START_WORDS = {
    "a", "an", "the", "this", "that", "these", "those", "its", "their", "our",
    "his", "her", "what", "why", "how", "when", "where", "who", "which",
    "beyond", "after", "before", "inside", "towards", "toward", "special",
}

HEADLINE_FRAGMENT_WORDS = {
    "biggest", "recent history", "financial fraud cases", "accounting enforcement",
    "enforcement roots", "new age", "special report", "next-gen", "next gen",
    "what the", "what is", "what are", "what means", "how to",
    "everything you need", "audit oversight",
}

INVALID_END_WORDS = {
    "history", "cases", "case", "report", "reports", "investigation",
    "investigations", "fraud", "corruption", "controversy", "lawsuit",
    "probe", "roots", "oversight",
}

CORPORATE_SUFFIXES = {
    "inc", "inc.", "incorporated", "corp", "corp.", "corporation", "company",
    "co", "co.", "ltd", "ltd.", "limited", "plc", "llc", "llp", "group",
    "holdings", "holding", "partners", "industries", "technologies", "technology",
    "pharmaceuticals", "bank", "airlines", "energy", "motors", "systems",
    "solutions", "international", "sa", "ag", "se", "nv", "n.v.", "spa",
    "s.p.a.", "gmbh", "oyj", "ab", "asa", "a/s",
}

CONTROVERSY_ACTION_WORDS = {
    "accused", "alleged", "allegedly", "investigated", "investigation",
    "investigating", "probed", "probe", "charged", "charges", "indicted",
    "convicted", "sued", "lawsuit", "fined", "penalty", "sanctioned",
    "sanctions", "violated", "violation", "breached", "breach", "recalled",
    "recall", "fraud", "bribery", "corruption", "misconduct", "pollution",
    "spill", "harassment", "discrimination", "forced labour", "forced labor",
    "child labour", "child labor", "money laundering", "data breach",
    "privacy violation", "greenwashing", "unsafe", "fatality", "fatalities",
}

In [47]:
# Text-cleaning and Google News RSS helpers
def clean_html_text(value):
    if value is None:
        return ""
    value = str(value)
    for _ in range(3):
        decoded = unescape(value)
        if decoded == value:
            break
        value = decoded
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = value.replace("\xa0", " ")
    return re.sub(r"\s+", " ", value).strip()


def remove_source_from_title(title, source):
    title = clean_html_text(title)
    source = clean_html_text(source)
    if not source:
        return title
    for suffix in [f" - {source}", f" – {source}", f" — {source}", f" | {source}"]:
        if title.lower().endswith(suffix.lower()):
            return title[:-len(suffix)].strip()
    return title


def collect_google_news_articles(search_queries):
    """Collect news results from Google News RSS without SerpAPI.

    Returns columns compatible with the rest of the notebook. A small retry
    policy handles transient failures. Persistent HTTP 429 stops the run to
    avoid hammering the service. Google News RSS availability and coverage
    are not guaranteed.
    """
    articles = []
    search_log = []
    stop_search = False
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "en-GB,en;q=0.9",
    })

    for query in search_queries:
        if stop_search:
            break

        rss_url = (
            "https://news.google.com/rss/search?q="
            + quote_plus(query)
            + f"&hl={quote_plus(GOOGLE_NEWS_LANGUAGE)}"
            + f"&gl={quote_plus(GOOGLE_NEWS_COUNTRY)}"
            + f"&ceid={quote_plus(GOOGLE_NEWS_CEID)}"
        )
        parsed_feed = None
        last_error = ""

        for attempt in range(MAX_RSS_RETRIES + 1):
            try:
                response = session.get(rss_url, timeout=REQUEST_TIMEOUT)
                if response.status_code == 429:
                    last_error = "HTTP 429 rate limited"
                    if attempt < MAX_RSS_RETRIES:
                        wait_seconds = RSS_RETRY_BASE_SECONDS * (2 ** attempt)
                        print(f"HTTP 429 for '{query}'. Waiting {wait_seconds}s before retry.")
                        time.sleep(wait_seconds)
                        continue
                    print(
                        "Persistent Google News RSS HTTP 429. Stopping discovery to "
                        "avoid repeated requests. Try again later or reduce query count."
                    )
                    stop_search = True
                    break

                response.raise_for_status()
                parsed_feed = feedparser.parse(response.content)
                if getattr(parsed_feed, "bozo", False) and not parsed_feed.entries:
                    last_error = str(getattr(parsed_feed, "bozo_exception", "RSS parse error"))
                    print(f"RSS parsing issue for '{query}': {last_error}")
                break
            except requests.exceptions.RequestException as error:
                last_error = f"{type(error).__name__}: {error}"
                if attempt < MAX_RSS_RETRIES:
                    wait_seconds = RSS_RETRY_BASE_SECONDS * (2 ** attempt)
                    print(f"Request error for '{query}'. Waiting {wait_seconds}s before retry.")
                    time.sleep(wait_seconds)
                else:
                    print(f"Skipping query '{query}' after request failure: {last_error}")

        count = 0
        if stop_search:
            search_log.append({
                "Search_Query": query,
                "Results_Returned": 0,
                "RSS_Error": last_error,
                "Rate_Limit_Stop": True,
            })
            break

        if parsed_feed is not None:
            entries = getattr(parsed_feed, "entries", []) or []
            for item in entries[:MAX_RESULTS_PER_QUERY]:
                title = clean_html_text(item.get("title", ""))
                summary = clean_html_text(item.get("summary", item.get("description", "")))
                link = item.get("link", "")
                source_obj = item.get("source", {}) or {}
                source_name = source_obj.get("title", "") if hasattr(source_obj, "get") else ""
                published = item.get("published", item.get("updated", ""))
                if not link or not title:
                    continue
                articles.append({
                    "Search_Query": query,
                    "Title": title,
                    "Summary": summary,
                    "Published": published,
                    "Source": clean_html_text(source_name),
                    "Link": link,
                })
                count += 1

        search_log.append({
            "Search_Query": query,
            "Results_Returned": count,
            "RSS_Error": last_error,
            "Rate_Limit_Stop": stop_search,
        })
        print(f"Query: {query} | RSS articles collected: {count}")

        if not stop_search:
            time.sleep(random.uniform(MIN_SEARCH_DELAY_SECONDS, MAX_SEARCH_DELAY_SECONDS))

    global rss_search_log
    rss_search_log = pd.DataFrame(search_log)
    return pd.DataFrame(
        articles,
        columns=["Search_Query", "Title", "Summary", "Published", "Source", "Link"],
    )


In [48]:
# Collect and prepare RSS articles
df_news = collect_google_news_articles(CONTROVERSY_SEARCH_QUERIES)

if df_news.empty:
    raise RuntimeError("No RSS records were returned. Check network access and Google News RSS availability.")

df_news = (
    df_news.drop_duplicates(subset=["Title", "Link"], keep="first")
    .reset_index(drop=True)
)
df_news["Article_ID"] = np.arange(1, len(df_news) + 1)
df_news["Clean_Title"] = df_news.apply(
    lambda row: remove_source_from_title(row["Title"], row["Source"]), axis=1
)
df_news["Published_Date"] = pd.to_datetime(df_news["Published"], errors="coerce", utc=True)

print(f"Unique RSS articles: {len(df_news):,}")
df_news[["Article_ID", "Clean_Title", "Source", "Published_Date"]].head(10)

# Inspect RSS query-level results, including any rate-limit stop.
print("\nRSS collection diagnostics:")
display(rss_search_log.head(20))


Query: "corporate fraud" | RSS articles collected: 10
Query: "accounting fraud" company | RSS articles collected: 10
Query: "money laundering" company | RSS articles collected: 10
Query: company bribery corruption | RSS articles collected: 10
Query: company regulatory investigation | RSS articles collected: 10
Query: company criminal charges | RSS articles collected: 10
Query: company lawsuit controversy | RSS articles collected: 10
Query: company antitrust investigation | RSS articles collected: 10
Query: company environmental violation | RSS articles collected: 10
Query: company pollution oil spill | RSS articles collected: 10
Query: company greenwashing investigation | RSS articles collected: 10
Query: company human rights violation | RSS articles collected: 10
Query: company forced labour | RSS articles collected: 10
Query: company child labour | RSS articles collected: 10
Query: company worker safety | RSS articles collected: 10
Query: company workplace fatality | RSS articles col

,Search_Query,Results_Returned,RSS_Error,Rate_Limit_Stop
0,"""corporate fraud""",10,,False
1,"""accounting fraud"" company",10,,False
2,"""money laundering"" company",10,,False
3,company bribery corruption,10,,False
4,company regulatory investigation,10,,False
5,company criminal charges,10,,False
6,company lawsuit controversy,10,,False
7,company antitrust investigation,10,,False
8,company environmental violation,10,,False
9,company pollution oil spill,10,,False


## Main article extraction

Google News RSS links are decoded to publisher URLs with `googlenewsdecoder` before requesting the article page. The extractor then tries Trafilatura and falls back to BeautifulSoup with common article-body selectors.

It records the publisher URL, HTTP status, extraction status, and article text. Some pages may still fail because of paywalls, JavaScript rendering, anti-bot controls, or unavailable content. Failed extractions fall back to the RSS title and summary for analysis; check `Text_Source` before relying on a result.


In [49]:
# Google News URL resolution
import re
from googlenewsdecoder import gnewsdecoder


def resolve_google_news_url(url):
    """Return a clean publisher URL from a Google News RSS link when possible."""
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()

    if "news.google.com" not in url.lower():
        return url

    try:
        result = gnewsdecoder(url, interval=1)

        if not isinstance(result, dict) or not result.get("success"):
            return None

        decoded_url = str(result.get("decoded_url") or "").strip()

        # The decoder may return Markdown in the form [label](destination URL).
        markdown_match = re.search(
            r"\]\((https?://[^)\s]+)\)",
            decoded_url
        )
        if markdown_match:
            return markdown_match.group(1).strip()

        # Also accept a plain URL.
        plain_match = re.fullmatch(r"(https?://\S+)", decoded_url)
        if plain_match:
            return plain_match.group(1).strip()

        # Last-resort: if a URL is embedded in a response string, extract it.
        embedded_match = re.search(r"(?<!\()https?://[^\s)\]]+", decoded_url)
        if embedded_match and "[" not in decoded_url and "](" not in decoded_url:
            return embedded_match.group(0).strip()

        return None

    except Exception as error:
        print(f"Google News decoding failed: {type(error).__name__}: {error}")
        return None


In [50]:
# Publisher article text extraction
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}


def extract_main_article_text(url):
    result = {
        "Resolved_URL": url,
        "Article_Text": "",
        "Extraction_Status": "",
        "HTTP_Status": None,
    }

    if not isinstance(url, str) or not url.strip():
        result["Extraction_Status"] = "Missing URL"
        return result

    try:
        original_url = url.strip()

        # Resolve Google News RSS links to publisher URLs before fetching HTML.
        resolved_url = resolve_google_news_url(original_url)
        article_url = resolved_url or original_url
        result["Resolved_URL"] = article_url

        response = requests.get(
            article_url,
            headers=REQUEST_HEADERS,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )

        result["HTTP_Status"] = response.status_code
        result["Resolved_URL"] = response.url

        print(
            f"HTTP {response.status_code} | "
            f"Final URL: {response.url[:180]}"
        )

        if response.status_code != 200:
            result["Extraction_Status"] = f"HTTP {response.status_code}"
            return result

        if "html" not in response.headers.get("Content-Type", "").lower():
            result["Extraction_Status"] = "Response is not HTML"
            return result

        # Try Trafilatura first.
        if USE_TRAFILATURA:
            try:
                extracted = trafilatura.extract(
                    response.text,
                    url=response.url,
                    include_comments=False,
                    include_tables=False,
                    favor_precision=False,
                )
                extracted = clean_html_text(extracted or "")

                if len(extracted) >= MIN_ARTICLE_LENGTH:
                    result["Article_Text"] = extracted
                    result["Extraction_Status"] = "Success (Trafilatura)"
                    return result

            except Exception as error:
                print(f"Trafilatura failed: {type(error).__name__}: {error}")

        # BeautifulSoup fallback.
        soup = BeautifulSoup(response.text, "html.parser")

        for element in soup.select(
            "script, style, noscript, nav, footer, header, aside, form, "
            "button, iframe, svg, figure, figcaption, "
            ".advertisement, .advert, .ads, .social-share, .related, "
            ".recommended, .newsletter, .cookie, .comments"
        ):
            element.decompose()

        article = soup.find("article")
        article_text = ""

        if article is not None:
            article_text = clean_html_text(
                " ".join(p.get_text(" ", strip=True) for p in article.find_all("p"))
            )

        # Try common article-body containers if <article> is absent or too short.
        if len(article_text) < MIN_ARTICLE_LENGTH:
            candidate_texts = []
            for selector in [
                '[itemprop="articleBody"]',
                ".article-body",
                ".article__body",
                ".story-body",
                ".story-content",
                ".entry-content",
                ".post-content",
                ".content-body",
                "main",
            ]:
                for container in soup.select(selector):
                    text = clean_html_text(
                        " ".join(p.get_text(" ", strip=True) for p in container.find_all("p"))
                    )
                    if text:
                        candidate_texts.append(text)

            if candidate_texts:
                article_text = max(candidate_texts, key=len)

        # Final fallback: all substantial paragraphs.
        if len(article_text) < MIN_ARTICLE_LENGTH:
            paragraphs = [
                clean_html_text(p.get_text(" ", strip=True))
                for p in soup.find_all("p")
            ]
            article_text = clean_html_text(
                " ".join(p for p in paragraphs if len(p) >= 40)
            )

        if len(article_text) >= MIN_ARTICLE_LENGTH:
            result["Article_Text"] = article_text
            result["Extraction_Status"] = "Success (BeautifulSoup)"
        else:
            result["Extraction_Status"] = "No substantial article text found"

        return result

    except requests.exceptions.Timeout:
        result["Extraction_Status"] = "Timeout"
    except requests.exceptions.SSLError:
        result["Extraction_Status"] = "SSL error"
    except requests.exceptions.RequestException as error:
        result["Extraction_Status"] = f"Request error: {type(error).__name__}"
    except Exception as error:
        result["Extraction_Status"] = (
            f"Extraction error: {type(error).__name__}: {error}"
        )

    return result


In [51]:
# Smoke-test the resolver and article extraction on one collected article before running the full extraction loop.

if not df_news.empty:
    test_url = df_news.iloc[0]["Link"]

    resolved = resolve_google_news_url(test_url)

    print("Original URL:", test_url[:180])
    print("Resolved URL:", repr(resolved))
    print("Resolution succeeded:", bool(resolved))

    if resolved:
        print(
            "Contains Markdown:",
            "[" in resolved or "](" in resolved
        )
        print(
            "Still a Google News URL:",
            "news.google.com" in resolved.lower()
        )
    else:
        print(
            "Resolution failed; the extraction function "
            "will fall back to the original URL."
        )

    # Extract the article text.
    test_result = extract_main_article_text(test_url)

    article_text = test_result.get("Article_Text", "") or ""

    print("\n--- Article Extraction Results ---")
    print("HTTP status:", test_result.get("HTTP_Status"))
    print("Extraction status:", test_result.get("Extraction_Status"))
    print("Final resolved URL:", test_result.get("Resolved_URL"))
    print("Extracted characters:", len(article_text))
    print("Extracted words:", len(article_text.split()))

    print("\n--- Extracted Article Text (first 1,500 characters) ---")
    if article_text:
        print(article_text[:1500])
    else:
        print("No article text was extracted.")

else:
    print("No RSS articles available to test.")

Original URL: https://news.google.com/rss/articles/CBMivAFBVV95cUxOb0kwZEFsOVdfcG5WY3JsOGJRUEtyZTZiRWZRa19MX1RleFEtUnJPMFVsWDJka2tNX2wwZGZmWkJLN1h2SUJ1clktanpuZVdkd3M0WWxfbzQ5R3UyYlNrZUxabkVuNW5
Resolved URL: 'https://www.ipswich.gov.uk/ipswich-newsroom/2026/ipswich-borough-council-corporate-fraud-service-helps-protect-more-ps51'
Resolution succeeded: True
Contains Markdown: False
Still a Google News URL: False
HTTP 200 | Final URL: https://www.ipswich.gov.uk/ipswich-newsroom/2026/ipswich-borough-council-corporate-fraud-service-helps-protect-more-ps51

--- Article Extraction Results ---
HTTP status: 200
Extraction status: Success (Trafilatura)
Final resolved URL: https://www.ipswich.gov.uk/ipswich-newsroom/2026/ipswich-borough-council-corporate-fraud-service-helps-protect-more-ps51
Extracted characters: 1919
Extracted words: 276

--- Extracted Article Text (first 1,500 characters) ---
Ipswich Borough Council’s dedicated Corporate Fraud Service has helped protect more than £5.1 million 

In [52]:
# Extract article text once per unique URL
article_extractions = []
unique_links = df_news["Link"].dropna().astype(str).drop_duplicates().tolist()

for article_number, link in enumerate(unique_links, start=1):
    result = extract_main_article_text(link)
    result["Original_Link"] = link
    article_extractions.append(result)
    print(f"{article_number}/{len(unique_links)} | {result['Extraction_Status']}")
    time.sleep(ARTICLE_DELAY_SECONDS)

df_extracted_text = pd.DataFrame(article_extractions)
df_news = df_news.merge(
    df_extracted_text, left_on="Link", right_on="Original_Link", how="left"
).drop(columns=["Original_Link"], errors="ignore")

df_news["Analysis_Text"] = df_news.apply(
    lambda row: clean_html_text(row.get("Article_Text", ""))
    if len(clean_html_text(row.get("Article_Text", ""))) >= MIN_ARTICLE_LENGTH
    else clean_html_text(f"{row.get('Clean_Title', '')}. {row.get('Summary', '')}"),
    axis=1,
)
df_news["Text_Source"] = np.where(
    df_news["Article_Text"].fillna("").str.len().ge(MIN_ARTICLE_LENGTH),
    "Full article",
    "RSS title and summary",
)

df_news["Extraction_Status"].value_counts(dropna=False)

HTTP 200 | Final URL: https://www.ipswich.gov.uk/ipswich-newsroom/2026/ipswich-borough-council-corporate-fraud-service-helps-protect-more-ps51
1/199 | Success (Trafilatura)
HTTP 405 | Final URL: https://www.eastsuffolk.gov.uk/news/2026/fraud-prevention-service-protects-millions-public-funds
2/199 | HTTP 405
HTTP 200 | Final URL: https://www.lw.com/en/insights/doj-fraud-division-announces-aggressive-approach-to-fight-corporate-fraud
3/199 | Success (Trafilatura)
HTTP 200 | Final URL: https://www.mayerbrown.com/en/insights/publications/2026/10/dojs-national-fraud-division-outlines-new-corporate-fraud-enforcement-priorities-and-factors
4/199 | Success (Trafilatura)
HTTP 200 | Final URL: https://www.hlc.com/en/publications/us-dojs-corporate-fraud-enforcement-framework-takes-shape
5/199 | Success (Trafilatura)
HTTP 200 | Final URL: https://www.morganlewis.com/pubs/2026/10/doj-provides-new-roadmap-for-corporate-fraud-enforcement
6/199 | Success (Trafilatura)
HTTP 202 | Final URL: https://www

KeyboardInterrupt: 

HTTP 200 is not an error. It means the web server successfully returned a response to your request. It does not necessarily mean your code has reached the publisher's website or extracted the article correctly.

In your controversy-monitor notebook, there are three separate things to check:
HTTP 200:	The requested page returned a successful HTTP response.
Resolved_URL:	Shows which URL was requested after the decoder ran and any redirects occurred.
Extraction_Status:	Tells you whether article text was successfully extracted.

Your earlier Mayer Brown test returned HTTP 200, Success (Trafilatura), and 13,808 extracted characters. That was a successful extraction, even though the HTTP status was 200.

## Controversy, tone, and company functions

Company involvement is scored using headline presence, repeated mentions, proximity to controversy language, grammatical subject position, and corporate-name suffixes. This is a heuristic ranking, not legal attribution.

In [ ]:
def contains_keyword(text, keyword):
    pattern = r"(?<!\w)" + re.escape(str(keyword).lower().strip()) + r"(?!\w)"
    return bool(re.search(pattern, str(text).lower(), flags=re.IGNORECASE))


SEVERITY_CANDIDATES = [
    "critical impact: catastrophic or widespread harm, fatalities, forced labour, human trafficking, or a major environmental disaster",
    "high impact: serious harm, major regulatory breach, substantial fraud, severe product safety failure, or major rights violation",
    "moderate impact: credible investigation, formal lawsuit, material allegation, regulatory probe, or significant operational harm",
    "low impact: limited complaint, criticism, early allegation, or minor compliance issue",
    "no clear material adverse impact described",
]

SEVERITY_LABEL_TO_SCORE = {
    SEVERITY_CANDIDATES[0]: 5,
    SEVERITY_CANDIDATES[1]: 4,
    SEVERITY_CANDIDATES[2]: 3,
    SEVERITY_CANDIDATES[3]: 2,
    SEVERITY_CANDIDATES[4]: 1,
}

THEME_CANDIDATES = {
    f"{theme} controversy": theme
    for theme in CONTROVERSY_DICTIONARY.keys()
}


def classify_model_theme_severity(text):
    """
    Zero-shot theme and impact classifier. Returns empty predictions if the
    model is unavailable. This is an assistive signal, not a verified rating.
    """
    if not USE_HF_ZERO_SHOT or zero_shot_pipeline is None:
        return {}, None, None

    sample = clean_html_text(text)[:2200]
    if not sample:
        return {}, None, None

    candidate_labels = list(THEME_CANDIDATES.keys()) + SEVERITY_CANDIDATES

    try:
        output = zero_shot_pipeline(
            sample,
            candidate_labels=candidate_labels,
            multi_label=True,
            truncation=True,
        )
        scores = dict(zip(output["labels"], output["scores"]))

        model_themes = {
            THEME_CANDIDATES[label]: float(score)
            for label, score in scores.items()
            if label in THEME_CANDIDATES
        }

        severity_scores = {
            label: float(scores.get(label, 0.0))
            for label in SEVERITY_CANDIDATES
        }
        best_severity_label = max(severity_scores, key=severity_scores.get)
        model_severity = SEVERITY_LABEL_TO_SCORE[best_severity_label]
        model_severity_confidence = severity_scores[best_severity_label]

        return model_themes, model_severity, model_severity_confidence

    except Exception as error:
        print(f"Zero-shot classification failed: {type(error).__name__}: {error}")
        return {}, None, None


def detect_controversies(text):
    """
    Combine auditable dictionary matches with zero-shot theme classification.
    Keyword severity is retained for audit; model severity is a separate signal.
    Final severity uses the model only when its top class clears the threshold.
    """
    results = []
    text = clean_html_text(text)
    if not text:
        return results

    model_themes, model_severity, model_severity_confidence = (
        classify_model_theme_severity(text)
    )

    for theme, severity_groups in CONTROVERSY_DICTIONARY.items():
        matched_by_severity = {}
        for severity, keywords in severity_groups.items():
            matches = [
                keyword for keyword in keywords
                if contains_keyword(text, keyword)
            ]
            if matches:
                matched_by_severity[severity] = matches

        keyword_severity = max(matched_by_severity) if matched_by_severity else None
        theme_model_score = model_themes.get(theme, 0.0)

        # Keep a theme if the dictionary matches it or the model identifies it
        # above the threshold. This can catch paraphrased reports.
        if keyword_severity is None and theme_model_score < ZERO_SHOT_THEME_THRESHOLD:
            continue

        use_model_severity = (
            model_severity is not None
            and model_severity_confidence is not None
            and model_severity_confidence >= ZERO_SHOT_SEVERITY_THRESHOLD
        )

        if use_model_severity:
            final_severity = model_severity
            severity_method = "Zero-shot model"
        elif keyword_severity is not None:
            final_severity = keyword_severity
            severity_method = "Keyword dictionary fallback"
        else:
            final_severity = 2
            severity_method = "Theme model; low severity confidence"

        results.append({
            "Theme": theme,
            "Severity": int(final_severity),
            "Keyword_Severity": keyword_severity,
            "Model_Severity": model_severity,
            "Model_Severity_Confidence": model_severity_confidence,
            "Theme_Model_Score": round(float(theme_model_score), 4),
            "Severity_Method": severity_method,
            "Matched_Keywords": sorted({
                keyword
                for matches in matched_by_severity.values()
                for keyword in matches
            }),
        })

    return results


def analyse_tone(text):
    """
    Return a signed tone score and label.
    Hugging Face multilingual sentiment is preferred; VADER is fallback.
    The score is a model sentiment signal, not a financial-impact score.
    """
    text = clean_html_text(text)
    if not text:
        return 0.0, "Unknown"

    if USE_HF_SENTIMENT and tone_pipeline is not None:
        try:
            prediction = tone_pipeline(
                text[:2500],
                truncation=True
            )[0]
            label = str(prediction.get("label", "neutral")).lower()
            confidence = float(prediction.get("score", 0.0))

            if "negative" in label:
                signed_score = -confidence
                tone_label = "Negative"
            elif "positive" in label:
                signed_score = confidence
                tone_label = "Positive"
            else:
                signed_score = 0.0
                tone_label = "Neutral"

            return round(signed_score, 4), tone_label

        except Exception as error:
            print(f"HF sentiment failed; trying VADER: {type(error).__name__}: {error}")

    if USE_VADER and sia is not None:
        score = sia.polarity_scores(text)["compound"]
        if score <= -0.60:
            label = "Very Negative"
        elif score <= -0.20:
            label = "Negative"
        elif score < 0.20:
            label = "Neutral"
        elif score < 0.60:
            label = "Positive"
        else:
            label = "Very Positive"
        return float(score), label

    return 0.0, "Unavailable"


def normalise_company_name(value):
    value = clean_html_text(value)
    value = re.sub(r"[’']s$", "", value, flags=re.IGNORECASE)
    value = value.strip(" \t\n\r.,;:!?()[]{}\"'“”‘’")
    return re.sub(r"\s+", " ", value)


def is_valid_company_candidate(candidate, source=""):
    candidate = normalise_company_name(candidate)
    source = normalise_company_name(source)
    candidate_lower, source_lower = candidate.lower(), source.lower()
    if not candidate or len(candidate) < MIN_COMPANY_CHARACTERS or candidate.isdigit():
        return False
    if "http" in candidate_lower or "www." in candidate_lower or "@" in candidate:
        return False
    if source_lower and candidate_lower == source_lower:
        return False
    source_without_domain = re.sub(r"\.(com|co\.uk|org|net|news)$", "", source_lower)
    if source_without_domain and candidate_lower == source_without_domain:
        return False
    if candidate_lower in NON_COMPANY_ORGANISATIONS:
        return False
    words = candidate.split()
    lower_words = candidate_lower.split()
    if len(words) > 7 or lower_words[0] in INVALID_COMPANY_START_WORDS:
        return False
    if any(fragment in candidate_lower for fragment in HEADLINE_FRAGMENT_WORDS):
        return False
    if lower_words[-1] in INVALID_END_WORDS:
        return False
    if len(words) >= 4 and sum(word.islower() for word in words) >= 2:
        return False
    return True


def has_corporate_suffix(company):
    company_words = set(re.findall(r"[A-Za-z.]+", company.lower()))
    return bool(company_words.intersection(CORPORATE_SUFFIXES))


def sentence_has_controversy(sentence_text):
    sentence_lower = sentence_text.lower()
    return any(keyword in sentence_lower for keyword in CONTROVERSY_ACTION_WORDS)


def identify_involved_companies(text, title="", source=""):
    """
    Identify company candidates using GLiNER when available, then score
    their involvement using headline presence, controversy proximity,
    repeated mentions, syntactic subject position and corporate suffixes.

    GLiNER NER confidence and involvement score are separate measures.
    NER confidence is not the probability that a company committed misconduct.
    """
    text = clean_html_text(text)
    title = clean_html_text(title)
    source = normalise_company_name(source)

    if not text:
        return []

    doc = nlp(text)

    company_data = defaultdict(lambda: {
        "Company": "",
        "Mentions": 0,
        "Controversy_Sentences": 0,
        "Headline_Mention": False,
        "Subject_of_Action": 0,
        "Corporate_Suffix": False,
        "NER_Confidence": None,
        "Evidence": [],
    })

    title_lower = title.lower()

    for sentence in doc.sents:
        sentence_raw = sentence.text
        sentence_text = clean_html_text(sentence_raw)
        if not sentence_text:
            continue

        sentence_is_controversial = sentence_has_controversy(sentence_text)
        entity_candidates = []

        # Preferred path: Hugging Face GLiNER with company-specific labels.
        # Fall back to spaCy only if GLiNER inference itself fails,
        # not when GLiNER successfully returns no company candidates.
        gliner_succeeded = False

        if USE_GLINER and company_ner is not None:
            try:
                predictions = company_ner.predict_entities(
                    sentence_raw,
                    GLINER_ENTITY_LABELS,
                    threshold=GLINER_THRESHOLD,
                )
                gliner_succeeded = True

                for prediction in predictions:
                    label = str(prediction.get("label", "")).strip().lower()

                    # Exclude regulators, government bodies, NGOs and media.
                    if label not in COMPANY_ENTITY_LABELS:
                        continue

                    candidate = prediction.get("text", "")
                    score = prediction.get("score", None)

                    start = prediction.get("start")
                    end = prediction.get("end")
                    token_span = None

                    if isinstance(start, int) and isinstance(end, int):
                        token_span = doc.char_span(
                            sentence.start_char + start,
                            sentence.start_char + end,
                            alignment_mode="expand",
                        )

                    entity_candidates.append({
                        "text": candidate,
                        "ner_score": float(score) if score is not None else None,
                        "token_span": token_span,
                    })

            except Exception as error:
                # Use spaCy for this sentence if GLiNER inference fails.
                print(
                    f"GLiNER inference failed for one sentence; "
                    f"using spaCy fallback. {type(error).__name__}: {error}"
                )
                entity_candidates = []

        # Fallback path: spaCy's general ORG entity label.
        if not USE_GLINER or company_ner is None or not gliner_succeeded:
            for entity in sentence.ents:
                if entity.label_ == "ORG":
                    entity_candidates.append({
                        "text": entity.text,
                        "ner_score": None,
                        "token_span": entity,
                    })

        # Aggregate each detected company candidate.
        for candidate_record in entity_candidates:
            company = normalise_company_name(candidate_record["text"])

            if not is_valid_company_candidate(company, source):
                continue

            key = company.casefold()
            data = company_data[key]

            if not data["Company"]:
                data["Company"] = company

            data["Mentions"] += 1

            if company.casefold() in title_lower.casefold():
                data["Headline_Mention"] = True

            if has_corporate_suffix(company):
                data["Corporate_Suffix"] = True

            ner_score = candidate_record["ner_score"]
            if ner_score is not None:
                previous_score = data["NER_Confidence"]
                data["NER_Confidence"] = (
                    ner_score if previous_score is None
                    else max(previous_score, ner_score)
                )

            if sentence_is_controversial:
                data["Controversy_Sentences"] += 1

                if len(data["Evidence"]) < 3:
                    data["Evidence"].append(sentence_text)

            token_span = candidate_record["token_span"]
            if (
                sentence_is_controversial
                and token_span is not None
                and any(
                    token.dep_ in {"nsubj", "nsubjpass"}
                    for token in token_span
                )
            ):
                data["Subject_of_Action"] += 1

    results = []

    for data in company_data.values():
        score = 0

        if data["Headline_Mention"]:
            score += 4

        score += min(data["Controversy_Sentences"] * 3, 9)
        score += min(data["Subject_of_Action"] * 2, 6)
        score += min(data["Mentions"], 5)

        if data["Corporate_Suffix"]:
            score += 2

        involvement_confidence = (
            "High" if score >= 10
            else "Medium" if score >= 6
            else "Low"
        )

        results.append({
            "Company": data["Company"],
            "Involvement_Score": score,
            "Confidence": involvement_confidence,
            "NER_Confidence": data["NER_Confidence"],
            "Mentions": data["Mentions"],
            "Controversy_Sentences": data["Controversy_Sentences"],
            "Headline_Mention": data["Headline_Mention"],
            "Evidence": data["Evidence"],
        })

    return sorted(
        results,
        key=lambda item: (
            item["Involvement_Score"],
            item["Mentions"]
        ),
        reverse=True,
    )


In [ ]:
# ------------------------------------------------------------
# 1. Prepare article text and cache repeated analyses
# ------------------------------------------------------------

start_time = time.perf_counter()

analysis_texts = (
    df_news["Analysis_Text"] # analysis_texts is a list containing all article texts
    .fillna("")
    .astype(str)
    .tolist()
)

print(f"Articles to analyse: {len(df_news):,}")
print(f"Unique article texts: {len(set(analysis_texts)):,}")


# Cache results for identical article text.
# This avoids repeating zero-shot classification for duplicate text.
controversy_cache = {}

def cached_detect_controversies(text):
    if text not in controversy_cache:
        controversy_cache[text] = detect_controversies(text)  # Analyse the text only if we haven't seen it before
    return controversy_cache[text]


df_news["Controversies"] = [
    cached_detect_controversies(text)
    for text in analysis_texts
]

df_news["Number_of_Themes"] = df_news["Controversies"].apply(len)

print(
    f"Controversy analysis completed in "
    f"{time.perf_counter() - start_time:.1f} seconds"
)


# ------------------------------------------------------------
# 2. Batch sentiment analysis
# ------------------------------------------------------------

tone_start = time.perf_counter()

def prediction_to_tone(prediction):
    """Convert one Hugging Face sentiment prediction to existing outputs."""
    if isinstance(prediction, list):
        prediction = prediction[0] if prediction else {}

    label = str(prediction.get("label", "neutral")).lower()
    confidence = float(prediction.get("score", 0.0))

    if "negative" in label:
        return round(-confidence, 4), "Negative"
    elif "positive" in label:
        return round(confidence, 4), "Positive"
    else:
        return 0.0, "Neutral"


tone_results = [None] * len(analysis_texts)

# Reuse results for repeated text.
tone_cache = {}
unique_positions = {}

for i, text in enumerate(analysis_texts):
    if text in tone_cache:
        tone_results[i] = tone_cache[text]
    else:
        unique_positions.setdefault(text, []).append(i)


if USE_HF_SENTIMENT and tone_pipeline is not None:
    # Only send non-empty, unique texts to the model.
    pending_texts = [
        text for text in unique_positions
        if clean_html_text(text)
    ]

    BATCH_SIZE = 16 # BATCH_SIZE = 16 tells the model to process up to 16 texts together

    try:
        for batch_start in range(0, len(pending_texts), BATCH_SIZE):
            batch_texts = pending_texts[
                batch_start:batch_start + BATCH_SIZE
            ]

            model_inputs = [
                clean_html_text(text)[:2500]
                for text in batch_texts
            ]

            predictions = tone_pipeline(
                model_inputs,
                batch_size=BATCH_SIZE,
                truncation=True,   # truncation=True and max_length=512 limit the input length according to the model's tokenizer
            )

            # Transformers versions may return either a dict per item
            # or a list of predictions per item.
            for text, prediction in zip(batch_texts, predictions):
                tone_cache[text] = prediction_to_tone(prediction)

            completed = min(
                batch_start + len(batch_texts),
                len(pending_texts)
            )

            print(
                f"Sentiment: {completed:,}/"
                f"{len(pending_texts):,} unique texts"
            )

    except Exception as error:
        print(
            "Batched sentiment failed; using the existing "
            f"analyse_tone fallback: {type(error).__name__}: {error}"
        )

# Fill results, using VADER / existing fallback if needed.
for text, positions in unique_positions.items():
    if text in tone_cache:
        result = tone_cache[text]
    elif not clean_html_text(text):
        result = (0.0, "Unknown")
    else:
        result = analyse_tone(text)

    tone_cache[text] = result

    for position in positions:
        tone_results[position] = result

# Assign the same output columns as before.
df_news["Tone_Score"] = [result[0] for result in tone_results]
df_news["Tone_Label"] = [result[1] for result in tone_results]

print(
    f"Tone analysis completed in "
    f"{time.perf_counter() - tone_start:.1f} seconds"
)


# ------------------------------------------------------------
# 3. Company identification with duplicate-result caching
# ------------------------------------------------------------

company_start = time.perf_counter()
company_cache = {}
company_assessments = []

for i, (_, row) in enumerate(df_news.iterrows(), start=1):
    text = str(row["Analysis_Text"]) if pd.notna(row["Analysis_Text"]) else ""
    title = str(row["Clean_Title"]) if pd.notna(row["Clean_Title"]) else ""
    source = str(row["Source"]) if pd.notna(row["Source"]) else ""

    cache_key = (text, title, source)

    if cache_key not in company_cache:
        company_cache[cache_key] = identify_involved_companies(
            text, title, source
        )

    company_assessments.append(company_cache[cache_key])

    if i % 25 == 0 or i == len(df_news):
        elapsed = time.perf_counter() - company_start
        print(
            f"Company identification: {i:,}/{len(df_news):,} "
            f"articles | {elapsed:.1f} seconds"
        )

df_news["Company_Assessments"] = company_assessments

df_news["Companies"] = df_news["Company_Assessments"].apply(
    lambda assessments: [
        item["Company"] for item in assessments
        if item["Involvement_Score"] >= MIN_INVOLVEMENT_SCORE
    ]
)

df_news["Number_of_Companies"] = df_news["Companies"].apply(len)

print(
    f"Company analysis completed in "
    f"{time.perf_counter() - company_start:.1f} seconds"
)

# Uses spaCy to process the article text
# Uses GLiNER to identify company names in sentences
# Scores the companies based on evidence, such as mentions in the headline and involvement in the controversy

# ------------------------------------------------------------
# 4. Preserve the downstream reporting structure
# ------------------------------------------------------------

df_matched = df_news[df_news["Number_of_Themes"] > 0].copy()

print(f"\nTotal analysis time: {time.perf_counter() - start_time:.1f} seconds")
print(f"Articles with dictionary/model matches: {len(df_matched):,}")
print(
    "Articles with selected companies: "
    f"{(df_matched['Number_of_Companies'] > 0).sum():,}"
)

display(
    df_matched[
        ["Clean_Title", "Companies", "Tone_Label", "Number_of_Themes"]
    ].head(20)
)


# Apply controversy, tone, and company analysis (original version)
df_news["Controversies"] = df_news["Analysis_Text"].apply(detect_controversies)
df_news["Number_of_Themes"] = df_news["Controversies"].apply(len)

_tone = df_news["Analysis_Text"].apply(analyse_tone)
df_news["Tone_Score"] = _tone.apply(lambda x: x[0])
df_news["Tone_Label"] = _tone.apply(lambda x: x[1])

df_news["Company_Assessments"] = df_news.apply(
    lambda row: identify_involved_companies(
        row["Analysis_Text"], row["Clean_Title"], row["Source"]
    ),
    axis=1,
)
df_news["Companies"] = df_news["Company_Assessments"].apply(
    lambda assessments: [
        item["Company"] for item in assessments
        if item["Involvement_Score"] >= MIN_INVOLVEMENT_SCORE
    ]
)
df_news["Number_of_Companies"] = df_news["Companies"].apply(len)

# Keep matched controversies for downstream company reporting.
df_matched = df_news[df_news["Number_of_Themes"] > 0].copy()

print(f"Articles with dictionary matches: {len(df_matched):,}")
print(f"Articles with selected companies: {(df_matched['Number_of_Companies'] > 0).sum():,}")
df_matched[["Clean_Title", "Companies", "Tone_Label", "Number_of_Themes"]].head(20)

## Optional LDA topic discovery

The controversy dictionary provides predefined, auditable themes. LDA adds data-driven topic words. It is skipped gracefully when there are too few articles or terms.

In [ ]:
# LDA topic discovery
topic_model_df = df_matched[["Article_ID", "Analysis_Text"]].drop_duplicates("Article_ID").reset_index(drop=True)

topic_model_df["Topic_ID"] = np.nan
topic_model_df["Topic_Confidence"] = np.nan
topic_model_df["Topic_Words"] = ""

if len(topic_model_df) >= 3:
    try:
        vectorizer = CountVectorizer(
            stop_words="english",
            min_df=2 if len(topic_model_df) >= 10 else 1,
            max_df=0.95 if len(topic_model_df) >= 10 else 1.0,
            ngram_range=(1, 2),
            max_features=5000,
        )
        X = vectorizer.fit_transform(topic_model_df["Analysis_Text"])
        effective_topic_count = min(NUMBER_OF_LDA_TOPICS, len(topic_model_df), X.shape[1])
        lda = LatentDirichletAllocation(
            n_components=effective_topic_count, random_state=42, learning_method="batch"
        )
        probabilities = lda.fit_transform(X)
        topic_model_df["Topic_ID"] = probabilities.argmax(axis=1) + 1
        topic_model_df["Topic_Confidence"] = probabilities.max(axis=1)
        feature_names = vectorizer.get_feature_names_out()
        topic_words = {}
        for topic_index, component in enumerate(lda.components_, start=1):
            top_indices = component.argsort()[-NUMBER_OF_TOPIC_WORDS:][::-1]
            words = [feature_names[index] for index in top_indices]
            topic_words[topic_index] = words
            print(f"Topic {topic_index}: {', '.join(words)}")
        topic_model_df["Topic_Words"] = topic_model_df["Topic_ID"].map(
            lambda topic_id: ", ".join(topic_words[int(topic_id)])
        )
    except ValueError as error:
        print(f"LDA skipped: {error}")
else:
    print("LDA skipped: fewer than three matched articles.")

df_matched = df_matched.merge(
    topic_model_df[["Article_ID", "Topic_ID", "Topic_Confidence", "Topic_Words"]],
    on="Article_ID", how="left"
)

In [ ]:
# Build detailed company assessment table
assessment_rows = []
for _, row in df_matched.iterrows():
    for assessment in row["Company_Assessments"]:
        assessment_rows.append({
            "Article_ID": row["Article_ID"],
            "Company": assessment["Company"],
            "Involvement_Score": assessment["Involvement_Score"],
            "Confidence": assessment["Confidence"],
            "NER_Confidence": assessment.get("NER_Confidence"),
            "Mentions": assessment["Mentions"],
            "Controversy_Sentences": assessment["Controversy_Sentences"],
            "Headline_Mention": assessment["Headline_Mention"],
            "Evidence": " | ".join(assessment["Evidence"]),
            "Selected": assessment["Involvement_Score"] >= MIN_INVOLVEMENT_SCORE,
            "Clean_Title": row["Clean_Title"],
            "Source": row["Source"],
            "Text_Source": row["Text_Source"],
            "Link": row["Link"],
        })

df_company_assessments = pd.DataFrame(assessment_rows)
if not df_company_assessments.empty:
    df_company_assessments = df_company_assessments.sort_values(
        ["Selected", "Involvement_Score", "Mentions"], ascending=[False, False, False]
    ).reset_index(drop=True)

df_company_assessments.head()

In [ ]:
# Expand to one selected company and controversy theme per row
selected_rows = []
for _, row in df_matched.iterrows():
    selected_assessments = [
        item for item in row["Company_Assessments"]
        if item["Involvement_Score"] >= MIN_INVOLVEMENT_SCORE
    ]
    for assessment in selected_assessments:
        for controversy in row["Controversies"]:
            selected_rows.append({
                "Article_ID": row["Article_ID"],
                "Company": assessment["Company"],
                "Involvement_Score": assessment["Involvement_Score"],
                "Company_Confidence": assessment["Confidence"],
                "Company_Mentions": assessment["Mentions"],
                "Evidence": " | ".join(assessment["Evidence"]),
                "Theme": controversy["Theme"],
                "Severity": controversy["Severity"],
                "Keyword_Severity": controversy.get("Keyword_Severity"),
                "Model_Severity": controversy.get("Model_Severity"),
                "Model_Severity_Confidence": controversy.get("Model_Severity_Confidence"),
                "Theme_Model_Score": controversy.get("Theme_Model_Score"),
                "Severity_Method": controversy.get("Severity_Method", ""),
                "Matched_Keywords": ", ".join(controversy["Matched_Keywords"]),
                "Tone_Score": row["Tone_Score"],
                "Tone_Label": row["Tone_Label"],
                "Topic_ID": row.get("Topic_ID", np.nan),
                "Topic_Confidence": row.get("Topic_Confidence", np.nan),
                "Topic_Words": row.get("Topic_Words", ""),
                "Clean_Title": row["Clean_Title"],
                "Summary": row["Summary"],
                "Source": row["Source"],
                "Published_Date": row["Published_Date"],
                "Text_Source": row["Text_Source"],
                "Extraction_Status": row["Extraction_Status"],
                "Link": row["Link"],
                "Resolved_URL": row["Resolved_URL"],
                "Search_Query": row["Search_Query"],
            })

article_results = pd.DataFrame(selected_rows)
SEVERITY_LABELS = {5: "Critical", 4: "High", 3: "Moderate", 2: "Emerging", 1: "Low", 0: "No match"}

if not article_results.empty:
    article_results["Severity_Label"] = article_results["Severity"].map(SEVERITY_LABELS).fillna("Unknown")
    article_results["Negative_Tone"] = (-article_results["Tone_Score"]).clip(lower=0, upper=1)
    article_results["Controversy_Score"] = (
        article_results["Severity"] * (1 + article_results["Negative_Tone"])
    ).round(3)
    article_results["Tone_Score"] = article_results["Tone_Score"].round(3)
    article_results["Topic_Confidence"] = article_results["Topic_Confidence"].round(3)
    article_results = article_results.sort_values(
        ["Controversy_Score", "Published_Date"], ascending=[False, False]
    ).reset_index(drop=True)

article_results.head()

In [ ]:
# Company and company-theme summaries
if article_results.empty:
    company_summary = pd.DataFrame()
    company_theme_summary = pd.DataFrame()
    print("No selected company-theme rows were created. Review thresholds and extraction results.")
else:
    company_summary = (
        article_results.groupby("Company", as_index=False)
        .agg(
            Articles=("Article_ID", "nunique"),
            Themes=("Theme", lambda x: ", ".join(sorted(set(x.dropna())))),
            Maximum_Severity=("Severity", "max"),
            Average_Severity=("Severity", "mean"),
            Average_Tone=("Tone_Score", "mean"),
            Maximum_Article_Score=("Controversy_Score", "max"),
            Total_Controversy_Score=("Controversy_Score", "sum"),
            Latest_Article_Date=("Published_Date", "max"),
        )
        .sort_values(["Total_Controversy_Score", "Maximum_Severity", "Articles"], ascending=False)
        .reset_index(drop=True)
    )

    company_theme_summary = (
        article_results.groupby(["Company", "Theme"], as_index=False)
        .agg(
            Articles=("Article_ID", "nunique"),
            Maximum_Severity=("Severity", "max"),
            Average_Severity=("Severity", "mean"),
            Average_Tone=("Tone_Score", "mean"),
            Total_Controversy_Score=("Controversy_Score", "sum"),
            Latest_Article_Date=("Published_Date", "max"),
        )
        .sort_values(["Total_Controversy_Score", "Maximum_Severity"], ascending=False)
        .reset_index(drop=True)
    )

    for frame in [company_summary, company_theme_summary]:
        for column in ["Average_Severity", "Average_Tone", "Maximum_Article_Score", "Total_Controversy_Score"]:
            if column in frame.columns:
                frame[column] = frame[column].round(3)

company_summary.head()

In [ ]:
# Export results to Excel
# Create export-safe copies
article_results_export = article_results.copy()
company_summary_export = company_summary.copy()
company_theme_summary_export = company_theme_summary.copy()
company_qc_export = df_company_assessments.copy()
matched_news_export = df_matched.copy()


def make_excel_safe(df):
    df = df.copy()

    for column in df.columns:

        # Convert timezone-aware datetime columns
        if isinstance(
            df[column].dtype,
            pd.DatetimeTZDtype
        ):
            df[column] = (
                df[column]
                .dt.tz_convert("UTC")
                .dt.tz_localize(None)
            )

        # Convert lists, dictionaries, sets and tuples to text
        elif df[column].dtype == "object":
            df[column] = df[column].apply(
                lambda value:
                str(value)
                if isinstance(
                    value,
                    (list, dict, set, tuple)
                )
                else value
            )

    return df

In [ ]:
article_results_export = make_excel_safe(
    article_results_export
)

company_summary_export = make_excel_safe(
    company_summary_export
)

company_theme_summary_export = make_excel_safe(
    company_theme_summary_export
)

company_qc_export = make_excel_safe(
    company_qc_export
)

matched_news_export = make_excel_safe(
    matched_news_export
)

In [ ]:
output_dir = Path("./controversy_monitor_exports")
output_dir.mkdir(parents=True, exist_ok=True)

workbook_path = output_dir / "controversy_monitor.xlsx"

with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:

    article_results_export.to_excel(
        writer,
        sheet_name="Article_Results",
        index=False
    )

    company_summary_export.to_excel(
        writer,
        sheet_name="Company_Summary",
        index=False
    )

    company_theme_summary_export.to_excel(
        writer,
        sheet_name="Company_Themes",
        index=False
    )

print(f"Workbook saved to: {workbook_path.resolve()}")

In [ ]:
output_dir = Path("./controversy_monitor_exports")

zip_path = shutil.make_archive(
    str(output_dir),
    "zip",
    root_dir=output_dir
)

print(f"ZIP created: {zip_path}")

In [ ]:
print("Excel files found:\n")

for f in Path(".").rglob("*.xlsx"):
    print(f.resolve())

## Review guidance

Before relying on the output:

- inspect **Company QC** for false-positive organisations;
- read the **Evidence** sentence for each selected company;
- adjust `MIN_INVOLVEMENT_SCORE` upward for more precision or downward for more recall;
- expand `NON_COMPANY_ORGANISATIONS` when regulators or NGOs appear;
- refine the controversy dictionary for your Responsible Investment framework;
- treat scores as screening signals, not findings of misconduct.

- **Tone is not severity**: sentiment models estimate positive/negative language, while severity models estimate described impact. Neither establishes guilt or verifies an allegation.
- Check `Text_Source`: title/snippet-only records are less reliable than full-article records.
- Review `Severity_Method`, `Keyword_Severity`, `Model_Severity_Confidence`, and the evidence sentences before using scores in research.
- Google News RSS may throttle frequent searches. Start with a small query subset before a full run.
